In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import timedelta
from hashlib import sha256
import json
from pathlib import Path
import re
import unicodedata

import lseg.data as ld
import pandas as pd


# ========= 設定 =========
@dataclass
class FetchConfig:
    query: str = "Language:LJA AND Source:RTRS"
    start: str = "2024-01-01T00:00:00Z"
    end: str = "2024-12-31T23:59:59Z"
    count: int = 100
    order_by: str = "new_to_old"
    chunk_days: int = 7
    cache_dir: str = "data/raw/headlines"
    force_refresh: bool = False


# ========= ヘルパー =========
def hash_values(values: list[str]) -> str:
    return sha256("||".join(values).encode("utf-8")).hexdigest()


def find_first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"候補カラムが見つかりません: {candidates}")


# ========= 正規化 =========
TIMESTAMP_CANDIDATES = ["versionCreated", "created", "published", "firstCreated", "date"]
HEADLINE_CANDIDATES = ["headline", "text", "storyHeadline", "headlineText", "title"]
STORY_ID_CANDIDATES = ["storyId", "story_id", "storyID", "id", "news_id"]
SOURCE_CANDIDATES = ["sourceCode", "source", "provider", "sourceName"]


def normalize_headlines_df(
    raw_df: pd.DataFrame,
    query: str = "",
    request_start: str | None = None,
    request_end: str | None = None,
    retrieved_at_utc: pd.Timestamp | None = None,
) -> pd.DataFrame:
    df = raw_df.copy()
    if df.empty:
        return pd.DataFrame(columns=[
            "news_id", "story_id", "timestamp", "headline", "source",
            "query", "request_start", "request_end", "retrieved_at_utc"
        ])

    # versionCreated index 優先
    if df.index.name == "versionCreated":
        df = df.reset_index()
        time_col = "versionCreated"
    elif isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        time_col = str(df.columns[0])
    else:
        df = df.reset_index()
        time_col = "versionCreated" if "versionCreated" in df.columns else find_first_existing_col(df, TIMESTAMP_CANDIDATES)

    headline_col = find_first_existing_col(df, HEADLINE_CANDIDATES)
    story_col = find_first_existing_col(df, STORY_ID_CANDIDATES)
    source_col = find_first_existing_col(df, SOURCE_CANDIDATES)

    out = pd.DataFrame()
    out["story_id"] = df[story_col].fillna("").astype(str)
    out["headline"] = df[headline_col].fillna("").astype(str)
    out["source"] = df[source_col].fillna("").astype(str)
    out["timestamp"] = pd.to_datetime(df[time_col], errors="coerce", utc=True)
    out = out.dropna(subset=["timestamp"]).reset_index(drop=True)

    out["query"] = query
    out["request_start"] = request_start
    out["request_end"] = request_end
    out["retrieved_at_utc"] = pd.to_datetime(retrieved_at_utc or pd.Timestamp.now("UTC"), utc=True)

    out["news_id"] = out.apply(
        lambda r: hash_values([str(r["story_id"]), str(r["timestamp"]), str(r["headline"]), str(r["source"])])[:20],
        axis=1,
    )

    return out[[
        "news_id", "story_id", "timestamp", "headline", "source",
        "query", "request_start", "request_end", "retrieved_at_utc"
    ]]


# ========= 取得 =========
def _resolve_order_by(order_by: str):
    return getattr(ld.news.SortOrder, order_by) if hasattr(ld.news.SortOrder, order_by) else order_by


def _make_chunk_windows(start: pd.Timestamp, end: pd.Timestamp, chunk_days: int):
    windows = []
    cursor = start
    while cursor < end:
        nxt = min(cursor + timedelta(days=chunk_days), end)
        windows.append((cursor, nxt))
        cursor = nxt
    return windows

def fetch_headlines(config: PipelineConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    """LSEGからヘッドラインを取得し、raw と正規化テーブルを返す。"""
    import lseg.data as ld

    fetch = config.lseg_fetch
    cache_dir = config.resolve_path(fetch.cache_dir)
    if cache_dir is None:
        raise ValueError("cache_dir が設定されていません。")
    cache_dir.mkdir(parents=True, exist_ok=True)

    start = pd.to_datetime(fetch.start, utc=True)
    end = pd.to_datetime(fetch.end, utc=True)
    if pd.isna(start) or pd.isna(end) or start >= end:
        raise ValueError("lseg_fetch.start/end の指定が不正です。")

    order_by = _resolve_order_by(fetch.order_by)
    chunk_windows = _make_chunk_windows(start, end, int(fetch.chunk_days))
    raw_frames: list[pd.DataFrame] = []
    normalized_frames: list[pd.DataFrame] = []

    session_opened = False
    try:
        # ここを追加
        ld.open_session()
        session_opened = True

        for chunk_start, chunk_end in chunk_windows:
            chunk_key = hash_values(
                [
                    fetch.query,
                    chunk_start.isoformat(),
                    chunk_end.isoformat(),
                    str(fetch.count),
                    str(fetch.order_by),
                ]
            )[:16]
            raw_path = cache_dir / f"headlines_raw_{chunk_key}.parquet"
            meta_path = cache_dir / f"headlines_raw_{chunk_key}.json"

            if raw_path.exists() and not fetch.force_refresh:
                chunk_raw = pd.read_parquet(raw_path)
                if "_index_versionCreated" in chunk_raw.columns:
                    chunk_raw = chunk_raw.set_index("_index_versionCreated")
                    chunk_raw.index.name = "versionCreated"
            else:
                try:
                    chunk_raw = ld.news.get_headlines(
                        query=fetch.query,
                        count=int(fetch.count),
                        start=chunk_start.to_pydatetime(),
                        end=chunk_end.to_pydatetime(),
                        order_by=order_by,
                    )
                except Exception as exc:
                    raise HeadlineFetchError(
                        "LSEG headlines 取得に失敗しました。認証情報とセッション設定を確認してください。"
                    ) from exc

                to_save = chunk_raw.copy()
                if isinstance(to_save.index, pd.DatetimeIndex) or to_save.index.name == "versionCreated":
                    index_name = to_save.index.name or "index"
                    to_save = to_save.reset_index().rename(columns={index_name: "_index_versionCreated"})
                raw_path.parent.mkdir(parents=True, exist_ok=True)
                to_save.to_parquet(raw_path, index=False)
                meta_path.write_text(
                    json.dumps(
                        {
                            "query": fetch.query,
                            "start": chunk_start.isoformat(),
                            "end": chunk_end.isoformat(),
                            "count": fetch.count,
                            "order_by": fetch.order_by,
                            "lseg_fetch": asdict(fetch),
                        },
                        ensure_ascii=False,
                        indent=2,
                    ),
                    encoding="utf-8",
                )

            raw_frames.append(chunk_raw)
            normalized_frames.append(
                normalize_headlines_df(
                    chunk_raw,
                    query=fetch.query,
                    request_start=chunk_start.isoformat(),
                    request_end=chunk_end.isoformat(),
                    retrieved_at_utc=pd.Timestamp.now("UTC"),
                )
            )
    finally:
        # ここを追加
        if session_opened:
            ld.close_session()

    if raw_frames:
        raw_all = pd.concat(raw_frames, axis=0)
    else:
        raw_all = pd.DataFrame()

    if normalized_frames:
        normalized_all = pd.concat(normalized_frames, ignore_index=True)
        normalized_all = normalized_all.sort_values("timestamp").drop_duplicates(subset=["news_id"], keep="first")
        normalized_all = normalized_all.reset_index(drop=True)
    else:
        normalized_all = normalize_headlines_df(pd.DataFrame())

    return raw_all, normalized_all



# ========= 前処理 =========
PREFIXES = [
    "UPDATE 1-", "UPDATE 2-", "UPDATE 3-", "RPT-", "BREAKING-", "BREAKINGVIEWS-",
    "BUZZ-", "訂正-", "再送-", "焦点：", "アングル：", "コラム：", "インタビュー：",
    "〔マーケットアイ〕", "〔需給情報〕",
]
LOW_INFORMATION_PATTERNS = ["東証前引け", "東証大引け", "今日の株式見通し", "午前の日経平均", "午後の日経平均", "外為市場", "需給情報"]
_URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
_SPACE_RE = re.compile(r"\s+")


def _strip_prefixes(text: str) -> str:
    out = text
    while True:
        changed = False
        for prefix in PREFIXES:
            if out.startswith(prefix):
                out = out[len(prefix):].lstrip(" -:：")
                changed = True
        if not changed:
            break
    return out


def _normalize_headline(text: str) -> str:
    out = unicodedata.normalize("NFKC", text)
    out = _URL_RE.sub(" ", out)
    out = _strip_prefixes(out)
    out = _SPACE_RE.sub(" ", out)
    return out.strip()


def _is_low_information(text: str) -> bool:
    return any(p in text for p in LOW_INFORMATION_PATTERNS)


def clean_headlines(df: pd.DataFrame, mode: str = "drop", min_chars: int = 8) -> pd.DataFrame:
    if mode not in {"drop", "flag"}:
        raise ValueError("mode は 'drop' または 'flag'")

    out = df.copy()
    out["headline"] = out["headline"].fillna("").astype(str)
    out["headline_clean"] = out["headline"].map(_normalize_headline)
    out["low_information_flag"] = out["headline_clean"].map(_is_low_information)
    out["too_short_flag"] = out["headline_clean"].str.len() < int(min_chars)

    out = out[~out["too_short_flag"]].copy()
    if mode == "drop":
        out = out[~out["low_information_flag"]].copy()
    return out.reset_index(drop=True)


# ========= 実行 =========
if __name__ == "__main__":
    cfg = FetchConfig()
    raw_df, normalized_df = fetch_headlines(cfg)
    cleaned_df = clean_headlines(normalized_df, mode="drop", min_chars=8)

    print("raw:", raw_df.shape)
    print("normalized:", normalized_df.shape)
    print("cleaned:", cleaned_df.shape)
